# 04 - Unweighted split conformal (the baseline that should fail)

Calibrate on the bright set, apply to the faint target. If bright and faint were exchangeable this would guarantee >=90% coverage at alpha=0.1 -- they aren't, by construction, so this is the headline failure case.

In [1]:
import numpy as np
import pandas as pd

cal = pd.read_parquet("../data/processed/cal_with_preds.parquet")
target = pd.read_parquet("../data/processed/target_with_preds.parquet")
ALPHA = 0.1

In [2]:
def nonconformity(z_true, z_pred):
    return np.abs(z_true - z_pred) / (1 + z_pred)  # scales error with redshift

scores = nonconformity(cal["z_spec"].values, cal["z_pred"].values)
n = len(scores)
q_level = min(np.ceil((n + 1) * (1 - ALPHA)) / n, 1.0)
q_hat = np.quantile(scores, q_level, method="higher")
q_hat

np.float64(0.160917858255524)

In [3]:
target["z_lower"] = target["z_pred"] - q_hat * (1 + target["z_pred"])
target["z_upper"] = target["z_pred"] + q_hat * (1 + target["z_pred"])
target["covered"] = (target["z_spec"] >= target["z_lower"]) & (target["z_spec"] <= target["z_upper"])

coverage = target["covered"].mean()
width = (target["z_upper"] - target["z_lower"]).mean()
print(f"nominal coverage: {1 - ALPHA:.2f}")
print(f"empirical coverage on faint target: {coverage:.3f}")
print(f"mean interval width: {width:.3f}")

nominal coverage: 0.90
empirical coverage on faint target: 0.429
mean interval width: 0.740


## Coverage by magnitude bin -- does the shortfall concentrate in the faintest objects?

In [4]:
target["mag_bin"] = pd.qcut(target["mag_F160W"], 4).astype(str)
target.groupby("mag_bin")["covered"].agg(["mean", "count"])

,mean,count
mag_bin,,
"(21.575, 22.053]",0.617706,497
"(22.053, 22.515]",0.489919,496
"(22.515, 23.027]",0.370968,496
"(23.027, 26.491]",0.237903,496


In [5]:
target.to_parquet("../data/processed/target_unweighted.parquet", index=False)
import json
json.dump(
    {"coverage_unweighted": float(coverage), "width_unweighted": float(width), "q_hat": float(q_hat)},
    open("../data/processed/unweighted_metrics.json", "w"),
)